# Expense Intelligence & Analytics Dashboard (Colab edition)

Run the cells **top to bottom** (Runtime -> Run all). No upload needed: every project file is written by the cells below.

1. Setup and write project files
2. Generate sample data and run tests
3. Explore the results in the notebook
4. (Optional) Launch the Streamlit dashboard through a public tunnel

## 1. Setup and project files

In [ ]:
!pip install -q -U "streamlit>=1.50"
!mkdir -p src data tests

In [ ]:
%%writefile src/__init__.py
# package marker

In [ ]:
%%writefile src/data_processing.py
"""Data loading and preprocessing for the Expense Intelligence Dashboard.

Responsibilities (Pandas + NumPy):
  * read a CSV of raw expenses
  * standardise column names and data types
  * clean messy amounts such as "Rs. 1,200" or "₹1200"
  * drop duplicates / invalid rows and report what was removed
  * add helper columns used by the analytics and dashboard layers
"""

from __future__ import annotations

import re
from typing import IO, Tuple, Union

import numpy as np
import pandas as pd

REQUIRED_COLUMNS = ["date", "description", "category", "amount"]


def _clean_amount(value) -> float:
    """Convert strings like '₹1,250.50' or 'Rs. 300' to a float. Bad values -> NaN."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)
    # Remove currency markers first so the "." in "Rs." doesn't survive,
    # then drop thousands separators and anything else that isn't numeric.
    text = re.sub(r"(?i)\b(rs|inr)\b\.?|₹", "", str(value))
    text = re.sub(r"[^\d.\-]", "", text)
    try:
        return float(text)
    except ValueError:
        return np.nan


def load_raw(source: Union[str, IO]) -> pd.DataFrame:
    """Read a CSV path or an uploaded file-like object."""
    return pd.read_csv(source)


def clean_expenses(raw: pd.DataFrame) -> Tuple[pd.DataFrame, dict]:
    """Clean the raw dataframe.

    Returns
    -------
    (clean_df, report)
        report is a dict with counts of what was fixed/removed, which the
        dashboard shows in a "Data quality" panel.
    """
    df = raw.copy()
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]

    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(
            f"Missing required column(s): {missing}. "
            f"Expected columns: {REQUIRED_COLUMNS}"
        )

    report = {"rows_in": int(len(df))}

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["amount"] = df["amount"].map(_clean_amount)

    bad_rows = df["date"].isna() | df["amount"].isna()
    report["invalid_date_or_amount"] = int(bad_rows.sum())
    df = df[~bad_rows]

    non_positive = df["amount"] <= 0
    report["non_positive_amount"] = int(non_positive.sum())
    df = df[~non_positive]

    before = len(df)
    df = df.drop_duplicates(subset=["date", "description", "amount"])
    report["duplicates_removed"] = int(before - len(df))

    df["description"] = df["description"].fillna("Unknown").astype(str).str.strip()
    df["category"] = (
        df["category"]
        .fillna("Uncategorized")
        .astype(str)
        .str.strip()
        .str.title()
        .replace("", "Uncategorized")
    )

    df["month"] = df["date"].dt.to_period("M").astype(str)
    df["weekday"] = df["date"].dt.day_name()

    df = df.sort_values("date").reset_index(drop=True)
    report["rows_out"] = int(len(df))
    return df, report

In [ ]:
%%writefile src/analytics.py
"""Analytics functions for the Expense Intelligence Dashboard.

All functions take the *clean* dataframe produced by data_processing.clean_expenses
and return small Pandas objects that the Streamlit layer can chart directly.
"""

from __future__ import annotations

from typing import Dict, Optional

import numpy as np
import pandas as pd


def summary_kpis(df: pd.DataFrame) -> dict:
    """Headline numbers shown at the top of the dashboard."""
    if df.empty:
        return {
            "total": 0.0,
            "transactions": 0,
            "avg_transaction": 0.0,
            "avg_monthly": 0.0,
            "top_category": "-",
        }
    monthly = df.groupby("month")["amount"].sum()
    by_cat = df.groupby("category")["amount"].sum()
    return {
        "total": float(df["amount"].sum()),
        "transactions": int(len(df)),
        "avg_transaction": float(df["amount"].mean()),
        "avg_monthly": float(monthly.mean()),
        "top_category": str(by_cat.idxmax()),
    }


def monthly_trend(df: pd.DataFrame) -> pd.DataFrame:
    """Total spend per month, plus month-over-month % change."""
    m = df.groupby("month", as_index=False)["amount"].sum().sort_values("month")
    m["mom_change_pct"] = m["amount"].pct_change() * 100
    return m.reset_index(drop=True)


def category_breakdown(df: pd.DataFrame) -> pd.DataFrame:
    """Spend per category with share of total."""
    c = (
        df.groupby("category", as_index=False)["amount"]
        .sum()
        .sort_values("amount", ascending=False)
        .reset_index(drop=True)
    )
    total = c["amount"].sum()
    c["share_pct"] = (c["amount"] / total * 100) if total else 0.0
    return c


def category_by_month(df: pd.DataFrame) -> pd.DataFrame:
    """Pivot table: rows = month, columns = category, values = spend."""
    return df.pivot_table(
        index="month", columns="category", values="amount", aggfunc="sum", fill_value=0.0
    ).sort_index()


def top_merchants(df: pd.DataFrame, n: int = 10) -> pd.DataFrame:
    """Top-n descriptions (merchants) by total spend."""
    return (
        df.groupby("description")
        .agg(total=("amount", "sum"), visits=("amount", "size"))
        .sort_values("total", ascending=False)
        .head(n)
        .reset_index()
    )


def weekday_spend(df: pd.DataFrame) -> pd.DataFrame:
    """Average transaction value per weekday, in calendar order."""
    order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    w = df.groupby("weekday")["amount"].agg(["sum", "mean", "count"]).reindex(order)
    return w.fillna(0.0).reset_index()


def detect_anomalies(
    df: pd.DataFrame, z_threshold: float = 3.0, min_samples: int = 5
) -> pd.DataFrame:
    """Flag unusually large transactions using a per-category z-score.

    z = (amount - category_mean) / category_std

    Categories with fewer than `min_samples` rows or zero variance are skipped,
    because a z-score is meaningless there.
    """
    if df.empty:
        return df.assign(z_score=pd.Series(dtype=float))

    grp = df.groupby("category")["amount"]
    mean = grp.transform("mean")
    std = grp.transform("std").replace(0, np.nan)
    count = grp.transform("count")

    z = (df["amount"] - mean) / std
    z = z.where(count >= min_samples)

    out = df.assign(z_score=z.round(2))
    out = out[out["z_score"] >= z_threshold]
    return out.sort_values("z_score", ascending=False).reset_index(drop=True)


def forecast_next_month(monthly: pd.DataFrame) -> Optional[float]:
    """Very simple linear-trend forecast of next month's total spend.

    Fits y = a*x + b over the monthly totals with numpy.polyfit (degree 1).
    Needs at least 3 months of data; otherwise returns None.
    """
    if len(monthly) < 3:
        return None
    y = monthly["amount"].to_numpy(dtype=float)
    x = np.arange(len(y), dtype=float)
    slope, intercept = np.polyfit(x, y, 1)
    return float(max(slope * len(y) + intercept, 0.0))


def budget_vs_actual(
    df: pd.DataFrame, budgets: Dict[str, float], month: str
) -> pd.DataFrame:
    """Compare a month's actual spend per category with a budget dict."""
    actual = df[df["month"] == month].groupby("category")["amount"].sum()
    rows = []
    for cat, budget in budgets.items():
        spent = float(actual.get(cat, 0.0))
        rows.append(
            {
                "category": cat,
                "budget": float(budget),
                "actual": spent,
                "remaining": float(budget) - spent,
                "used_pct": (spent / budget * 100) if budget else np.nan,
            }
        )
    return pd.DataFrame(rows)

In [ ]:
%%writefile data/generate_sample_data.py
"""Generate a realistic, slightly messy sample expense dataset.

Run:  python data/generate_sample_data.py
Output: data/sample_expenses.csv

The data is synthetic (seeded, so it is reproducible). It deliberately includes
messy values so the cleaning step in src/data_processing.py has real work to do:
  * amounts written as "Rs. 1,200" / "₹450"
  * a few missing amounts and invalid dates
  * a duplicated row
  * a handful of genuine outliers (to be caught by the anomaly detector)
"""

from pathlib import Path

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# category -> (merchants, mean amount, std, avg transactions per month)
CATEGORIES = {
    "Food": (["Swiggy", "Zomato", "Campus Canteen", "Starbucks", "Dominos"], 350, 120, 22),
    "Groceries": (["BigBasket", "DMart", "Nature's Basket"], 900, 300, 6),
    "Transport": (["Uber", "Ola", "Namma Metro", "BMTC Bus"], 140, 60, 18),
    "Shopping": (["Amazon", "Flipkart", "Myntra", "Decathlon"], 1400, 600, 4),
    "Entertainment": (["Netflix", "BookMyShow", "Spotify", "Steam"], 450, 200, 4),
    "Utilities": (["Airtel Recharge", "Electricity Bill", "Jio Fiber"], 750, 150, 3),
    "Education": (["Coursera", "Udemy", "Stationery Store", "Xerox Shop"], 650, 350, 3),
    "Health": (["Apollo Pharmacy", "Practo", "MedPlus"], 500, 250, 2),
}

rows = []
for month_start in pd.date_range("2025-10-01", "2026-09-01", freq="MS"):
    days_in_month = month_start.days_in_month
    for category, (merchants, mean, std, per_month) in CATEGORIES.items():
        n = max(1, int(rng.poisson(per_month)))
        for _ in range(n):
            day = int(rng.integers(1, days_in_month + 1))
            amount = max(20.0, float(rng.normal(mean, std)))
            rows.append(
                {
                    "date": month_start.replace(day=day).strftime("%Y-%m-%d"),
                    "description": str(rng.choice(merchants)),
                    "category": category,
                    "amount": round(amount, 2),
                }
            )

df = pd.DataFrame(rows)

# --- inject outliers -------------------------------------------------------
outliers = [
    ("2026-01-18", "Amazon", "Shopping", 14999.00),
    ("2026-03-07", "Apollo Pharmacy", "Health", 6200.00),
    ("2026-05-22", "Zomato", "Food", 3400.00),
    ("2026-08-14", "Uber", "Transport", 2650.00),
]
df = pd.concat(
    [df, pd.DataFrame(outliers, columns=["date", "description", "category", "amount"])],
    ignore_index=True,
)

# --- make the data messy ---------------------------------------------------
df["amount"] = df["amount"].astype(object)
idx = rng.choice(df.index, size=12, replace=False)
for i, k in enumerate(idx):
    val = float(df.at[k, "amount"])
    df.at[k, "amount"] = f"Rs. {val:,.2f}" if i % 2 == 0 else f"₹{val:,.0f}"

df.loc[rng.choice(df.index, size=3, replace=False), "amount"] = np.nan
df.loc[rng.choice(df.index, size=2, replace=False), "date"] = "not a date"
df.loc[rng.choice(df.index, size=4, replace=False), "category"] = "  food "
df = pd.concat([df, df.sample(3, random_state=1)], ignore_index=True)  # duplicates

df = df.sample(frac=1, random_state=7).reset_index(drop=True)

out_path = Path(__file__).parent / "sample_expenses.csv"
df.to_csv(out_path, index=False)
print(f"Wrote {len(df)} rows to {out_path}")

In [ ]:
%%writefile tests/test_analytics.py
"""Basic tests. Run with:  python -m pytest -q   (or just: python tests/test_analytics.py)"""

import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from src import analytics as an  # noqa: E402
from src.data_processing import clean_expenses  # noqa: E402


def _raw():
    return pd.DataFrame(
        {
            "Date": ["2026-01-05", "2026-01-05", "2026-02-10", "bad", "2026-02-11", "2026-03-01"],
            "Description": ["Swiggy", "Swiggy", "Uber", "Amazon", "Zomato", "Netflix"],
            "Category": ["food", "food", " transport", "Shopping", None, "Entertainment"],
            "Amount": ["Rs. 1,200.50", "Rs. 1,200.50", "₹300", "500", np.nan, 650],
        }
    )


def test_cleaning_report_and_values():
    clean, rep = clean_expenses(_raw())
    assert rep["rows_in"] == 6
    assert rep["invalid_date_or_amount"] == 2  # "bad" date + NaN amount
    assert rep["duplicates_removed"] == 1
    assert rep["rows_out"] == 3
    assert clean["amount"].dtype == float
    assert set(clean["category"]) == {"Food", "Transport", "Entertainment"}
    assert clean.loc[clean["description"] == "Swiggy", "amount"].iloc[0] == 1200.50


def test_missing_column_raises():
    try:
        clean_expenses(pd.DataFrame({"date": [1], "amount": [2]}))
    except ValueError as exc:
        assert "Missing required" in str(exc)
    else:
        raise AssertionError("expected ValueError")


def test_anomaly_detection_flags_outlier():
    amounts = [100, 110, 95, 105, 98, 102, 107, 99, 5000]
    df = pd.DataFrame(
        {
            "date": pd.date_range("2026-01-01", periods=len(amounts)),
            "description": "x",
            "category": "Food",
            "amount": amounts,
        }
    )
    df["month"] = df["date"].dt.to_period("M").astype(str)
    flagged = an.detect_anomalies(df, z_threshold=2.0)
    assert len(flagged) == 1
    assert flagged.iloc[0]["amount"] == 5000


def test_forecast_linear_trend():
    monthly = pd.DataFrame({"month": ["a", "b", "c", "d"], "amount": [100, 200, 300, 400]})
    assert round(an.forecast_next_month(monthly)) == 500
    assert an.forecast_next_month(monthly.head(2)) is None


def test_budget_vs_actual():
    df = pd.DataFrame(
        {"month": ["2026-01", "2026-01"], "category": ["Food", "Food"], "amount": [300.0, 200.0]}
    )
    out = an.budget_vs_actual(df, {"Food": 400, "Travel": 100}, "2026-01")
    food = out[out["category"] == "Food"].iloc[0]
    assert food["actual"] == 500 and food["remaining"] == -100


if __name__ == "__main__":
    for name, fn in list(globals().items()):
        if name.startswith("test_"):
            fn()
            print("PASS", name)

In [ ]:
%%writefile app.py
"""Expense Intelligence & Analytics Dashboard (Streamlit).

Run:  streamlit run app.py
"""

from pathlib import Path

import pandas as pd
import streamlit as st

from src import analytics as an
from src.data_processing import REQUIRED_COLUMNS, clean_expenses, load_raw

SAMPLE_PATH = Path(__file__).parent / "data" / "sample_expenses.csv"

st.set_page_config(page_title="Expense Intelligence", page_icon="💸", layout="wide")
st.title("💸 Expense Intelligence & Analytics Dashboard")
st.caption("Upload your expenses, clean them with Pandas/NumPy, and explore spending patterns.")


# ---------------------------------------------------------------- data loading
@st.cache_data(show_spinner=False)
def prepare(raw: pd.DataFrame):
    return clean_expenses(raw)


with st.sidebar:
    st.header("1. Data")
    uploaded = st.file_uploader("Upload expenses CSV", type=["csv"])
    st.caption(f"Required columns: {', '.join(REQUIRED_COLUMNS)}")
    use_sample = st.checkbox("Use sample data", value=uploaded is None)

try:
    if uploaded is not None and not use_sample:
        raw = load_raw(uploaded)
    elif SAMPLE_PATH.exists():
        raw = load_raw(SAMPLE_PATH)
    else:
        st.error("Sample data missing. Run `python data/generate_sample_data.py` first.")
        st.stop()
    clean, report = prepare(raw)
except ValueError as exc:
    st.error(str(exc))
    st.stop()

if clean.empty:
    st.warning("No valid rows after cleaning.")
    st.stop()

# ---------------------------------------------------------------------- filters
with st.sidebar:
    st.header("2. Filters")
    min_d, max_d = clean["date"].min().date(), clean["date"].max().date()
    date_range = st.date_input("Date range", (min_d, max_d), min_value=min_d, max_value=max_d)
    cats = sorted(clean["category"].unique())
    chosen = st.multiselect("Categories", cats, default=cats)
    z_thr = st.slider("Anomaly sensitivity (z-score)", 2.0, 5.0, 3.0, 0.5)

if isinstance(date_range, tuple) and len(date_range) == 2:
    start, end = date_range
else:
    start, end = min_d, max_d

df = clean[
    (clean["date"].dt.date >= start)
    & (clean["date"].dt.date <= end)
    & (clean["category"].isin(chosen))
]
if df.empty:
    st.warning("No data for the selected filters.")
    st.stop()

# ------------------------------------------------------------------------- KPIs
k = an.summary_kpis(df)
c1, c2, c3, c4, c5 = st.columns(5)
c1.metric("Total spend", f"₹{k['total']:,.0f}")
c2.metric("Transactions", f"{k['transactions']:,}")
c3.metric("Avg / transaction", f"₹{k['avg_transaction']:,.0f}")
c4.metric("Avg / month", f"₹{k['avg_monthly']:,.0f}")
c5.metric("Top category", k["top_category"])

tab_over, tab_cat, tab_anom, tab_budget, tab_data = st.tabs(
    ["Trends", "Categories", "Anomalies", "Budget", "Data"]
)

# ----------------------------------------------------------------------- Trends
with tab_over:
    monthly = an.monthly_trend(df)
    st.subheader("Monthly spend")
    st.line_chart(monthly.set_index("month")["amount"])

    forecast = an.forecast_next_month(monthly)
    if forecast is not None:
        st.info(f"📈 Linear-trend forecast for next month: **₹{forecast:,.0f}**")
    else:
        st.caption("Need at least 3 months of data for a forecast.")

    left, right = st.columns(2)
    with left:
        st.subheader("Average spend by weekday")
        wk = an.weekday_spend(df).set_index("weekday")["mean"]
        st.bar_chart(wk)
    with right:
        st.subheader("Month-over-month change (%)")
        st.dataframe(
            monthly.assign(mom_change_pct=monthly["mom_change_pct"].round(1)),
            width="stretch",
            hide_index=True,
        )

# ------------------------------------------------------------------- Categories
with tab_cat:
    cat = an.category_breakdown(df)
    left, right = st.columns(2)
    with left:
        st.subheader("Spend by category")
        st.bar_chart(cat.set_index("category")["amount"])
    with right:
        st.subheader("Share of total")
        st.dataframe(
            cat.assign(share_pct=cat["share_pct"].round(1)),
            width="stretch",
            hide_index=True,
        )
    st.subheader("Category spend per month")
    st.area_chart(an.category_by_month(df))
    st.subheader("Top merchants")
    st.dataframe(an.top_merchants(df, 10), width="stretch", hide_index=True)

# ---------------------------------------------------------------------- Anomalies
with tab_anom:
    st.subheader("Unusually large transactions")
    st.caption(
        "A transaction is flagged when its amount is far above the average for its "
        "own category: z = (amount - category mean) / category std."
    )
    anomalies = an.detect_anomalies(df, z_threshold=z_thr)
    if anomalies.empty:
        st.success("No anomalies at this sensitivity.")
    else:
        st.warning(f"{len(anomalies)} transaction(s) flagged")
        st.dataframe(
            anomalies[["date", "description", "category", "amount", "z_score"]],
            width="stretch",
            hide_index=True,
        )

# ------------------------------------------------------------------------ Budget
with tab_budget:
    st.subheader("Budget vs actual")
    months = sorted(df["month"].unique(), reverse=True)
    month = st.selectbox("Month", months)
    st.caption("Set a monthly budget for each category:")
    budgets = {}
    cols = st.columns(4)
    for i, c in enumerate(sorted(df["category"].unique())):
        default = int(round(df.groupby("category")["amount"].sum()[c] / df["month"].nunique(), -2))
        budgets[c] = cols[i % 4].number_input(c, min_value=0, value=max(default, 100), step=100)
    bva = an.budget_vs_actual(df, budgets, month)
    st.dataframe(bva.round(1), width="stretch", hide_index=True)
    over = bva[bva["remaining"] < 0]
    if not over.empty:
        st.error("Over budget: " + ", ".join(over["category"]))

# --------------------------------------------------------------------------- Data
with tab_data:
    st.subheader("Data quality report")
    st.json(report)
    st.subheader("Cleaned data")
    st.dataframe(df, width="stretch", hide_index=True)
    st.download_button(
        "Download cleaned CSV",
        df.to_csv(index=False).encode("utf-8"),
        file_name="cleaned_expenses.csv",
        mime="text/csv",
    )

## 2. Generate sample data and run tests

In [ ]:
!python data/generate_sample_data.py
!python tests/test_analytics.py

## 3. Explore the results

### Cleaning (Pandas + NumPy)

In [ ]:
import sys
sys.path.insert(0, ".")
import pandas as pd
import matplotlib.pyplot as plt
from src.data_processing import load_raw, clean_expenses
from src import analytics as an

raw = load_raw("data/sample_expenses.csv")
clean, report = clean_expenses(raw)
print("Data quality report:", report)
print("KPIs:", an.summary_kpis(clean))
clean.head()

### Monthly trend and forecast

In [ ]:
monthly = an.monthly_trend(clean)
ax = monthly.plot(x="month", y="amount", marker="o", figsize=(10, 4), legend=False, title="Monthly spend")
ax.set_ylabel("Rupees")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print(f"Linear-trend forecast for next month: Rs. {an.forecast_next_month(monthly):,.0f}")

### Spend by category

In [ ]:
cat = an.category_breakdown(clean)
cat.plot.barh(x="category", y="amount", figsize=(8, 4), legend=False, title="Spend by category").invert_yaxis()
plt.tight_layout()
plt.show()
cat.round(1)

### Anomalies (per-category z-score)

In [ ]:
an.detect_anomalies(clean, z_threshold=3.0)[["date", "description", "category", "amount", "z_score"]]

## 4. (Optional) Launch the Streamlit dashboard

Colab has no public URL of its own, so this uses a free `localtunnel`:

1. Run the cell below. It prints an **IP address**, which is the tunnel password.
2. Click the `https://....loca.lt` link it prints.
3. Paste the IP as the password on the page that opens.

The cell keeps running while the app is live. Stop it with the square button when done.

In [ ]:
!streamlit run app.py --server.port 8501 --server.headless true &>/content/streamlit_logs.txt &
import time; time.sleep(5)
print("Tunnel password (your IP):")
!curl -s ipv4.icanhazip.com
!npx -y localtunnel --port 8501